# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronitjoshi7/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page is worth reviewing first if many people already see it, if it has not been updated in a while, and if it ranks where better content or a better title could move it up. I turn that into a 0-100 score by adding four simple parts with fixed weights, and nothing is fitted to the label. Each page also gets a reason code that says why it scored.

The details: 40% visibility (how high the page's impressions rank among all pages), 30% staleness (days since last update divided by 365, capped at 1), 25% position opportunity (average position 4-10 scores 1.0, 11-20 scores 0.7, 1-3 scores 0.3, 21+ scores 0.1, no position data scores 0), and 5% thinness (1 if word count is under 1,200). The four weights come from the dataset guide's starter baseline. The component definitions are my own simple choices, and I set them before looking at results.

Reason codes the rule can output: stale_visible_page, thin_visible_page, page_one_decay_risk, low_ctr_visible_page, low_engagement_visible_page. A page gets the first one that applies, in that order. I left out the guide's declining_with_demand code because it uses trend_direction, which is my label.

How many pages each code fires on (see the code below): stale_visible_page and thin_visible_page almost never fire (0.1% and 0.3%), while the other three fire on 24-33% of pages. 13,392 pages get no reason code at all. The thresholds are policy choices, and some clearly don't fit this file well.

In [8]:
# This cell is for CODE (numbers, a query, a check).
import os
import numpy as np
import pandas as pd

path = next((p for p in ["data/raw/content_refresh_anonymized.csv",
                         "content_refresh_anonymized.csv"] if os.path.exists(p)), None)
if path is None:
    raise FileNotFoundError("Upload content_refresh_anonymized.csv using the folder icon in Colab.")
df = pd.read_csv(path)

imp, ses, pos = df["impressions_90d"], df["sessions_90d"], df["avg_position"]
codes = {
    "stale_visible_page": (df["days_since_last_update"] >= 180) & (imp >= 500),
    "thin_visible_page": (df["word_count"] > 0) & (df["word_count"] < 1200) & (imp >= 250),
    "page_one_decay_risk": (pos > 0) & (pos <= 10) & (df["content_age_days"] >= 180),
    "low_ctr_visible_page": (imp >= 500) & (pos > 0) & (pos <= 20) & (df["ctr"] < 0.5),
    "low_engagement_visible_page": (ses >= 30) & ((df["engagement_rate"] < 30) | (df["scroll_rate"] < 30)),
}
flags = pd.DataFrame(codes)
for name in flags:
    print(f"{name}: fires on {flags[name].sum():,} pages ({flags[name].mean() * 100:.1f}%)")
print("pages with no reason code:", f"{(flags.sum(axis=1) == 0).sum():,}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


stale_visible_page: fires on 17 pages (0.1%)
thin_visible_page: fires on 82 pages (0.3%)
page_one_decay_risk: fires on 7,076 pages (23.6%)
low_ctr_visible_page: fires on 9,759 pages (32.5%)
low_engagement_visible_page: fires on 7,113 pages (23.7%)
pages with no reason code: 13,392


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I score all 30,000 pages, sort from highest to lowest, and write the ranked list to work/outputs/baseline_action_score.csv with the columns rank, content_id, client_id, baseline_score, reason_code, n_reason_codes, suggested_action, confidence. Each reason code maps to one action: refresh content, expand content, review and protect a page-one page, rewrite title and meta, or improve on-page engagement. Pages with no code get "monitor." Confidence is "medium" if a page has two or more codes and 500+ impressions, "low" if it has one code, and "very low" if it has none. This is a rough label for how many signals agree, not a measured probability. The score reads only measured 90-day fields. It never reads the label or its source columns.

In [9]:
# This cell is for CODE (numbers, a query, a check).
weights = {"visibility": 0.40, "staleness": 0.30, "position": 0.25, "thin": 0.05}

visibility = df["impressions_90d"].rank(pct=True)
staleness = (df["days_since_last_update"] / 365).clip(0, 1)
position = np.select([pos == 0, pos <= 3, pos <= 10, pos <= 20], [0.0, 0.3, 1.0, 0.7], default=0.1)
thin = ((df["word_count"] > 0) & (df["word_count"] < 1200)).astype(float)

score_inputs = ["impressions_90d", "days_since_last_update", "avg_position", "word_count",
                "content_age_days", "ctr", "sessions_90d", "engagement_rate", "scroll_rate"]

df["baseline_score"] = 100 * (weights["visibility"] * visibility + weights["staleness"] * staleness
                              + weights["position"] * position + weights["thin"] * thin)

actions = {"stale_visible_page": "refresh content",
           "thin_visible_page": "expand content",
           "page_one_decay_risk": "review and protect page-one page",
           "low_ctr_visible_page": "rewrite title and meta",
           "low_engagement_visible_page": "improve on-page engagement"}
df["n_reason_codes"] = flags.sum(axis=1)
df["reason_code"] = flags.apply(lambda r: next((c for c in flags.columns if r[c]), "none"), axis=1)
df["suggested_action"] = df["reason_code"].map(actions).fillna("monitor")
df["confidence"] = np.where((df["n_reason_codes"] >= 2) & (df["impressions_90d"] >= 500), "medium",
                            np.where(df["n_reason_codes"] >= 1, "low", "very low"))

queue = df.sort_values("baseline_score", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1
cols = ["rank", "content_id", "client_id", "baseline_score", "reason_code",
        "n_reason_codes", "suggested_action", "confidence"]
os.makedirs("work/outputs", exist_ok=True)
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote", len(queue), "rows to work/outputs/baseline_action_score.csv")
print("score range:", round(queue["baseline_score"].min(), 1), "to", round(queue["baseline_score"].max(), 1))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


wrote 30000 rows to work/outputs/baseline_action_score.csv
score range: 0.8 to 73.5


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

All 20 top pages look alike. They are keyword articles in the page_1 position tier with very high impressions (about 149,000 to 518,000 in 90 days). 13 get the page_one_decay_risk code (action: review and protect a page-one page) and 7 get low_ctr_visible_page (action: rewrite title and meta). All 20 have "medium" confidence, which only means two or more signals agree. They come from just 3 clients, and 17 of the 20 are from a single client. All 20 have exactly the same days_since_last_update (104), so staleness cannot separate them. Their scores sit between 73.47 and 73.55, so the order within the top 20 is close to arbitrary.

What would make these wrong: for page_one_decay_risk, the page is old but stable and not losing visibility. For low_ctr_visible_page, a low click rate is normal for this kind of query and isn't a title problem. The table below lists a "wrong if" line for each page.

In [10]:
# This cell is for CODE (numbers, a query, a check).
wrong_if = {
    "page_one_decay_risk": "page is old but stable, not actually losing visibility",
    "low_ctr_visible_page": "low CTR is normal for this kind of query, not a title problem",
    "stale_visible_page": "page is old but still accurate and performing",
    "thin_visible_page": "short page fully answers the query",
    "low_engagement_visible_page": "engagement is measured differently here (rates are on different systems)",
}
top20 = queue.head(20).copy()
top20["wrong_if"] = top20["reason_code"].map(wrong_if)
print(top20["reason_code"].value_counts().to_dict())
top20[["rank", "content_id", "baseline_score", "reason_code", "suggested_action", "confidence", "wrong_if"]].round(1)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


{'page_one_decay_risk': 13, 'low_ctr_visible_page': 7}


,rank,content_id,baseline_score,reason_code,suggested_action,confidence,wrong_if
0,1,content_5fe46e04994d,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."
1,2,content_2c2606c5d176,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."
2,3,content_cb112fce36be,73.5,low_ctr_visible_page,rewrite title and meta,medium,"low CTR is normal for this kind of query, not ..."
3,4,content_36ff89c8214e,73.5,low_ctr_visible_page,rewrite title and meta,medium,"low CTR is normal for this kind of query, not ..."
4,5,content_c21024970297,73.5,low_ctr_visible_page,rewrite title and meta,medium,"low CTR is normal for this kind of query, not ..."
5,6,content_c8e9d6ab9013,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."
6,7,content_d17681677e69,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."
7,8,content_a7427266c305,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."
8,9,content_3d94572c3a35,73.5,low_ctr_visible_page,rewrite title and meta,medium,"low CTR is normal for this kind of query, not ..."
9,10,content_01908772c6db,73.5,page_one_decay_risk,review and protect page-one page,medium,"page is old but stable, not actually losing vi..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Leakage check: none of the columns the score reads is the label or one of its source columns (the check below returns an empty set). The label is used only to check the ranking afterwards, never to build the score.

Same slice as the model later: the starter filter keeps all 30,000 rows, so this baseline is evaluated on the same rows and the same label a model will use. When I compare them, I'll use the same test clients.

The picks look weak. In this data 54.2% of pages are labeled "down," so random picking would get about 54% right. The baseline gets 45% right in its top 20 (9 of 20), 38% in its top 50, 35% in its top 100, and 47% in its top 500. That is below the base rate at every cutoff I checked, and "always predict down" scores 0.542 on its own.

Weak picks in the top 20: 11 are not labeled "down." All 11 are labeled "stable," but their impressions changed by -6.3% at the median, from -19.6% to +14.6%. So some are mild decliners that fall just short of the -20% cutoff, which means the label's cutoff, a policy choice, partly decides what counts as wrong. Also, 17 of the 20 come from one client, and all 20 are page-1 keyword articles with very high impressions, so the list is narrow.

My guess is that the 40% visibility weight pushes the highest-impression pages to the top, and those are less often labeled "down." I haven't tested that. I am not going to re-tune the weights to make the numbers look better, because tuning on the label I evaluate against would just fit this file. The baseline stays frozen now that model work is next. Any change should be tested later on clients the rule was not tuned on.

In [11]:
# This cell is for CODE (numbers, a query, a check).
label_cols = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
              "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d"]
print("score inputs that are label or window columns:", set(score_inputs) & set(label_cols))

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# the label is used ONLY here, to check the ranking, never to build the score
labels = (queue["trend_direction"] == "down").astype(int)
base = labels.mean()
for k in (20, 50, 100, 500):
    p = precision_at_k(queue["baseline_score"], labels, k)
    print(f"precision@{k}: {p:.2f} (base rate {base:.2f})")

t = queue.head(20)
print("top 20 labeled down:", int(labels.head(20).sum()), "of 20")
print("top 20 not labeled down, by trend_direction:", t.loc[labels.head(20) == 0, "trend_direction"].value_counts().to_dict())
print("top 20 clients:", t["client_id"].nunique(), "| position tiers:", t["position_tier"].value_counts().to_dict())
print("top 20 days_since_last_update:", t["days_since_last_update"].unique().tolist())
print("top 20 impressions_90d: min", t["impressions_90d"].min(), "| max", t["impressions_90d"].max())

from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="most_frequent").fit(queue[["impressions_90d"]], labels)
print("dummy (always predict 'down') accuracy:", round(dummy.score(queue[["impressions_90d"]], labels), 3))
print("not-down top-20 pages, change in impressions (%):", t.loc[labels.head(20) == 0, "trend_pct"].describe().round(1)[["min", "50%", "max"]].to_dict())
print("top 20 pages per client:", sorted(t["client_id"].value_counts().tolist(), reverse=True))
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


score inputs that are label or window columns: set()
precision@20: 0.45 (base rate 0.54)
precision@50: 0.38 (base rate 0.54)
precision@100: 0.35 (base rate 0.54)
precision@500: 0.47 (base rate 0.54)
top 20 labeled down: 9 of 20
top 20 not labeled down, by trend_direction: {'stable': 11}
top 20 clients: 3 | position tiers: {'page_1': 20}
top 20 days_since_last_update: [104]
top 20 impressions_90d: min 148737 | max 517715
dummy (always predict 'down') accuracy: 0.542
not-down top-20 pages, change in impressions (%): {'min': -19.6, '50%': -6.3, 'max': 14.6}
top 20 pages per client: [17, 2, 1]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.